<a href="https://colab.research.google.com/github/m7-code/AI_Models/blob/main/text_to_video_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python imageio imageio-ffmpeg gTTS moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
import imageio
from IPython.display import Video, display
from gtts import gTTS
from moviepy.editor import VideoFileClip, AudioFileClip
import os

print("⏳ Model load ho raha hai (pehli baar ~5GB download hoga, baad mein temporary cache se uthayega)...")
adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-2",
    torch_dtype=torch.float16
)
pipeline = AnimateDiffPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    motion_adapter=adapter,
    torch_dtype=torch.float16
)
pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)
pipeline.to("cuda")
print("✅ Model ready!")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning

⏳ Model load ho raha hai (pehli baar ~5GB download hoga, baad mein temporary cache se uthayega)...


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

...

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✅ Model ready!


In [ ]:
video_prompt = "A professional male news anchor in his 30s wearing a formal dark navy blue suit coat with a crisp white dress shirt and a solid red tie, standing in a modern TV news studio with a solid blue screen background. He is speaking directly to the camera with a confident, engaging expression, one hand gesturing toward an invisible weather map as he delivers the weather forecast. His tone is informative yet friendly, pointing and explaining weather conditions. Professional broadcast studio lighting, sharp focus, photorealistic, 8K resolution, live weather news segment style"  # <-- APNA PROMPT YAHAN LIKHEN
print(f"🎬 Generating video for: '{video_prompt}'")

output = pipeline(
    prompt=video_prompt,
    num_frames=32,               # 16 frames ≈ 2 sec (8 fps)
    num_inference_steps=30,      # 20 steps, balance speed/quality
    guidance_scale=7.5,
    generator=torch.manual_seed(42)
)
frames = output.frames[0]

# Frames se silent video banayein
silent_video_path = "silent_video.mp4"
imageio.mimsave(silent_video_path, frames, fps=8)
print(f"✅ Silent video save ho gayi: {silent_video_path}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (107 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['tone is informative yet friendly , pointing and explaining weather conditions . professional broadcast studio lighting , sharp focus , photorealistic , 8 k resolution , live weather news segment style']


🎬 Generating video for: 'A professional male news anchor in his 30s wearing a formal dark navy blue suit coat with a crisp white dress shirt and a solid red tie, standing in a modern TV news studio with a solid blue screen background. He is speaking directly to the camera with a confident, engaging expression, one hand gesturing toward an invisible weather map as he delivers the weather forecast. His tone is informative yet friendly, pointing and explaining weather conditions. Professional broadcast studio lighting, sharp focus, photorealistic, 8K resolution, live weather news segment style'


  0%|          | 0/30 [00:00<?, ?it/s]

✅ Silent video save ho gayi: silent_video.mp4


In [ ]:
voice_text = "Aaj ka mausam mulk ke aksar hisson mein saaf aur khushgawar rahega. Lahore, Karachi aur Islamabad mein din bhar dhoop rahegi, jabke paharon par halki baarish ka imkaan hai. Hararat 25 se 32 darje centigrade ke darmiyan rahegi"  # <-- APNA AUDIO TEXT YAHAN LIKHEN
lang_code = "ur"  # 'hi'=Hindi, 'ur'=Urdu, 'en'=English (jo chahiye badle)

tts = gTTS(text=voice_text, lang=lang_code, slow=False)
voiceover_path = "voiceover.mp3"
tts.save(voiceover_path)
print(f" Voiceover save ho gaya: {voiceover_path}")


 Voiceover save ho gaya: voiceover.mp3


In [ ]:
video_clip = VideoFileClip(silent_video_path)
audio_clip = AudioFileClip(voiceover_path)

# Agar audio chhoti hai to loop karein, ya video duration ke hisaab se trim karein
if audio_clip.duration < video_clip.duration:
    audio_clip = audio_clip.loop(duration=video_clip.duration)
else:
    audio_clip = audio_clip.subclip(0, video_clip.duration)

final_video = video_clip.set_audio(audio_clip)
final_video_path = "final_video_with_voice.mp4"
final_video.write_videofile(final_video_path, codec='libx264', audio_codec='aac', fps=8)
print(f"✅ Final video tayar hai: {final_video_path}")

Moviepy - Building video final_video_with_voice.mp4.
MoviePy - Writing audio in final_video_with_voiceTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video final_video_with_voice.mp4



Moviepy - Done !
Moviepy - video ready final_video_with_voice.mp4
✅ Final video tayar hai: final_video_with_voice.mp4


In [ ]:
print(" Ab aapki video with voice play ho rahi hai:")
display(Video(final_video_path, embed=True, width=640))


 Ab aapki video with voice play ho rahi hai:
